# 052 — RAG & tool use

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/052-rag-and-tool-use.ipynb)

Retrieval picks text for the context window; tool use picks actions outside it. Here you rank chunks by cosine, then pack the best ones into a token budget — the two steps that decide what the model actually sees.

## 1. Rank three chunks with cosine

Cosine is the dot product divided by both lengths. With unit-length vectors it is just the dot product, so the arithmetic stays visible.

In [ ]:
import numpy as np                              # array library, preinstalled on Colab

query = np.array([0.6, 0.8])                    # embedded question
chunks = {                                      # name -> (vector, token count)
    "A": (np.array([0.6, 0.8]), 400),
    "B": (np.array([0.8, 0.6]), 500),
    "C": (np.array([-0.6, 0.8]), 300),
}

for name, (vec, tokens) in chunks.items():      # one line per chunk
    cos = query @ vec / (np.linalg.norm(query) * np.linalg.norm(vec))
    print(f"{name}: cos={cos:.2f}  tokens={tokens}")   # A 1.00, B 0.96, C 0.28

## 2. Pack the budget

Ranking is not packing. A high-ranked chunk can be too big, and a smaller one below it may still fit. Admit in rank order, skip what does not fit, and stop when the budget is spent.

In [ ]:
budget = 1100                                   # tokens available for chunks
ranked = [("A", 400), ("B", 500), ("C", 300), ("D", 200)]   # by cosine

selected, remaining = [], budget                # start empty, full budget
for name, tokens in ranked:                     # walk down the ranking
    if tokens <= remaining:                     # only admit what fits whole
        selected.append(name)                   # keep the chunk id
        remaining -= tokens                     # spend its tokens
    else:
        print(f"skip {name}: {tokens} > {remaining}")   # too big, keep going

print("selected:", selected, "remaining:", remaining)   # ['A','B','D'] 0

## Your turn

Set `budget = 700` and predict the selected set before rerunning. Then add a chunk `E` with 150 tokens and cosine 0.10 — does it enter the prompt, and does that match your intuition about relevance?